In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from trl import SFTTrainer, SFTConfig
import json

from peft import LoraConfig, TaskType
from peft import AutoPeftModelForCausalLM

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# model to be used
model_name_phi_3_128k = 'microsoft/Phi-3-mini-128k-instruct'

model_name = model_name_phi_3_128k
#custom model name
new_model = 'phi-3_finetuned_lora_2_Epochs'

In [ ]:
# lora parameters according to the original model repository

# 'lora_r' is the dimension of the LoRA attention.
lora_r = 16

# 'lora_alpha' is the alpha parameter for LoRA scaling.
lora_alpha = 16

# 'lora_dropout' is the dropout probability for LoRA layers.
lora_dropout = 0.05

# 'target_modules' is a list of the modules in the model that will be replaced with LoRA layers.
target_modules= ['k_proj', 'q_proj', 'v_proj', 'o_proj', "gate_proj", "down_proj", "up_proj"]

# 'set_seed' is a function that sets the seed for generating random numbers, 
# which is used for reproducibility of the results.
set_seed(42)

In [ ]:
#loading model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2")
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.padding_side = 'right'

In [ ]:
# loading dataset
train_dataset = load_dataset('zjunlp/OceanInstruct', split='train')

In [ ]:
# function to format the train dataset entries in the chat format to be used by the model
def filter_and_format_examples(examples):
    inputs = examples['en_input']
    outputs = examples['en_output']
    formatted_texts = []
    for inp, out in zip(inputs, outputs):
        formatted_text = f"User: {inp + tokenizer.eos_token}\nAssistant: {out + tokenizer.eos_token}"
        formatted_texts.append(formatted_text)
    return {'messages': formatted_texts}
    

In [ ]:
train_dataset

In [ ]:
train_dataset = train_dataset.map(filter_and_format_examples, batched=True)

In [ ]:
train_dataset = train_dataset.remove_columns(['task_type', 'en_input', 'en_output', 'cn_input', 'cn_output'])
train_dataset

In [ ]:
eval_dataset = load_dataset('zjunlp/OceanBench', split='train')
eval_dataset

In [ ]:
# function to format the eval dataset entries to the chat format
def parse_and_format_eval_examples(examples):
    formatted_texts = []
    for entry in examples['text']:
        try:
            data = json.loads(entry)
            inp = data.get('input', '')
            out = data.get('output', '')
            formatted_text = f"User: {inp + tokenizer.eos_token}\nAssistant: {out + tokenizer.eos_token}"
            formatted_texts.append(formatted_text)
        except json.JSONDecodeError as e:
            print(f"JSON decode error: {e}")
            formatted_texts.append(f"User: \nAssistant: ")
    return {'messages': formatted_texts}


In [ ]:
eval_dataset = eval_dataset.map(parse_and_format_eval_examples, batched=True)
eval_dataset = eval_dataset.remove_columns(['text'])
eval_dataset

In [ ]:
# Training config

args = SFTConfig(
    output_dir=f"G:\\Masterarbeit\\{new_model}",
    evaluation_strategy="steps", #steps
    do_eval=True,
    eval_on_start=True,
    optim="adamw_torch",            # optimizer according to the original model repository
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    log_level="debug",
    save_strategy="steps",
    save_steps=125, #125
    logging_strategy="steps",
    logging_steps=125, #125
    learning_rate=1e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    eval_steps=125, #125
    num_train_epochs=2,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",     # learning rate scheduler according to the original model repository
    seed=42,
    dataset_text_field="messages",
    max_seq_length=512,
)

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=args,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model()

In [ ]:
# loading lora model setup and combining it with the original to save the full model

new_model = AutoPeftModelForCausalLM.from_pretrained(
    args.output_dir,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.bfloat16, #torch.float16,
    trust_remote_code=True,
    # device_map=device_map,
)

merged_model = new_model.merge_and_unload()

merged_model.save_pretrained("G:\\Masterarbeit\\merged_model", trust_remote_code=True, safe_serialization=True)

tokenizer.save_pretrained("G:\\Masterarbeit\\merged_model")
